# Stock Market Prediction System - Phase 3: Model Creation
**Academic College ML Project SOP**

### Core Requirement:
- Implement **Linear Regression** as the regression baseline.
- Implement **Custom Decision Tree Classifier** manually from scratch using **Gini Index** (WITHOUT using scikit-learn for the custom tree).

### Why Decision Tree + Gini Impurity?
1. Decision Trees handle non-linear market feature interactions effectively without assuming Gaussian normality.
2. Provides interpretable, transparent decision rules directly traceable during college evaluation.
3. Gini Impurity: $Gini = 1 - \sum_{i=1}^C (p_i)^2$. Minimizing Gini yields the purest partition of Bullish/Bearish states.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Chronological Train/Test Split (80% Train / 20% Test - No Shuffling)
train_size = int(len(clean_dataset) * 0.8)
train = clean_dataset.iloc[:train_size]
test = clean_dataset.iloc[train_size:]

feature_cols = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'daily_return', 'return_3d', 'return_5d', 'return_10d',
    'SMA_5', 'SMA_10', 'SMA_20', 'SMA_50',
    'volatility_5', 'volatility_10', 'volatility_20',
    'high_low_ratio', 'close_open_ratio', 'price_range',
    'volume_change', 'momentum_5', 'momentum_10'
]

X_train_raw = train[feature_cols].values
X_test_raw = test[feature_cols].values

# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

print(f'Training samples: {len(train)}, Testing samples: {len(test)}')

### 1. Baseline Linear Regression (Predicting Tomorrow High and Low)

In [ ]:
lr_high = LinearRegression()
lr_high.fit(X_train_scaled, train['target_high'].values)

lr_low = LinearRegression()
lr_low.fit(X_train_scaled, train['target_low'].values)

pred_high = lr_high.predict(X_test_scaled)
pred_low = lr_low.predict(X_test_scaled)

print('Linear Regression training complete.')
print('Sample Test Prediction High (First 5):', np.round(pred_high[:5], 2))
print('Sample Test Actual High (First 5):    ', np.round(test['target_high'].values[:5], 2))

### 2. Manual Custom Decision Tree Classifier (Gini Impurity from Scratch)

In [ ]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, *, value=None, probabilities=None, samples_count=0):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value
        self.probabilities = probabilities
        self.samples_count = samples_count

    @property
    def is_leaf(self):
        return self.value is not None

class CustomDecisionTreeClassifier:
    def __init__(self, max_depth=5, min_samples_split=5, min_samples_leaf=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.root = None
        self.classes_ = np.array([])

    def _gini(self, y):
        if len(y) == 0: return 0.0
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1.0 - np.sum(p ** 2)

    def _best_split(self, X, y):
        best_gini, best_feat, best_thresh = float('inf'), None, None
        n_samples, n_features = X.shape
        if n_samples < self.min_samples_split:
            return None, None
        for feat in range(n_features):
            vals = np.unique(X[:, feat])
            if len(vals) <= 1: continue
            threshs = np.percentile(vals, np.linspace(5, 95, 20)) if len(vals) > 20 else (vals[:-1] + vals[1:]) / 2.0
            for t in threshs:
                left = X[:, feat] <= t
                nl, nr = np.sum(left), n_samples - np.sum(left)
                if nl < self.min_samples_leaf or nr < self.min_samples_leaf: continue
                wgini = (nl / n_samples) * self._gini(y[left]) + (nr / n_samples) * self._gini(y[~left])
                if wgini < best_gini:
                    best_gini, best_feat, best_thresh = wgini, feat, t
        return best_feat, best_thresh

    def _build(self, X, y, depth=0):
        classes = np.unique(y)
        if depth >= self.max_depth or len(classes) == 1 or len(y) < self.min_samples_split:
            vals, counts = np.unique(y, return_counts=True)
            probs = {c: float(counts[list(vals).index(c)]/len(y)) if c in vals else 0.0 for c in self.classes_}
            return Node(value=vals[np.argmax(counts)], probabilities=probs, samples_count=len(y))
        feat, thresh = self._best_split(X, y)
        if feat is None:
            vals, counts = np.unique(y, return_counts=True)
            probs = {c: float(counts[list(vals).index(c)]/len(y)) if c in vals else 0.0 for c in self.classes_}
            return Node(value=vals[np.argmax(counts)], probabilities=probs, samples_count=len(y))
        left = X[:, feat] <= thresh
        return Node(feature_idx=feat, threshold=thresh, left=self._build(X[left], y[left], depth+1), right=self._build(X[~left], y[~left], depth+1), samples_count=len(y))

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.root = self._build(np.asarray(X), np.asarray(y), 0)
        return self

    def _predict_one(self, node, x):
        if node.is_leaf: return node.value
        return self._predict_one(node.left if x[node.feature_idx] <= node.threshold else node.right, x)

    def predict(self, X):
        return np.array([self._predict_one(self.root, x) for x in np.asarray(X)])

# Fit Custom Decision Tree for Market Direction (Bullish / Bearish)
custom_tree = CustomDecisionTreeClassifier(max_depth=5)
custom_tree.fit(X_train_raw, train['direction_target'].values)

dir_preds = custom_tree.predict(X_test_raw)
print('Custom Decision Tree trained successfully!')
print('Sample Predictions:', dir_preds[:10])